# Verify background clips are frog-free

Listen to each 3 s clip in `data/raw/background/` and mark whether any frog or toad is audible.
Answers are written to `labels/background_verification.csv` (tracked in git, so commit it when done).
Clips marked `yes`/`unsure` are excluded from the negative (background) class; you can resume any time.

In [ ]:
from pathlib import Path

import pandas as pd
import soundfile as sf

from anuran.config import ROOT

BG = ROOT / "data" / "raw" / "background"
LABELS = ROOT / "labels"
LABELS.mkdir(exist_ok=True)

rows = []
for f in sorted(BG.glob("*.wav")):
    info = sf.info(f)
    rows.append({
        "clip_id": f.stem,
        "audio_path": str(f),
        "category": f.stem.rsplit("_", 1)[0],  # e.g. traffic_traffic_1521 -> traffic_traffic
        "start_time": 0.0,
        "end_time": round(info.duration, 3),
        "sample_rate": info.samplerate,
    })
clips = pd.DataFrame(rows)
clips_path = ROOT / "data" / "processed" / "review" / "background_clips.csv"
clips_path.parent.mkdir(parents=True, exist_ok=True)
clips.to_csv(clips_path, index=False)
print(len(clips), "clips")
print(clips["sample_rate"].value_counts().to_dict())
clips["category"].value_counts()

In [ ]:
from jupyter_bioacoustic import BioacousticAnnotator

form = {
    "title": {"value": "Background clip: any frog or toad audible?", "progress_tracker": True},
    "form": [
        {"select": {"column": "frog_present", "label": "Frog/toad audible in [[category]]?",
                    "required": True, "items": ["no", "yes", "unsure"]}},
        {"textbox": {"column": "notes", "label": "Notes (species if known)"}},
    ],
    "submission_buttons": {"line": True, "next": {"label": "Skip", "icon": True},
                           "submit": {"label": "Submit", "icon": True}},
}

ja = BioacousticAnnotator(
    data=str(clips_path),
    data_index_column="clip_id",
    data_start_time="start_time",
    data_end_time="end_time",
    audio_column="audio_path",
    display_columns=["category", "clip_id", "sample_rate"],
    form_config=form,
    output=str(LABELS / "background_verification.csv"),
    default_buffer=0,
    sort="category",
)
ja.open()

In [ ]:
# progress
out = LABELS / "background_verification.csv"
if out.exists():
    done = pd.read_csv(out)
    print(f"{done['clip_id'].nunique()} / {len(clips)} reviewed")
    print(done["frog_present"].value_counts().to_dict())